# VRAM Wall — self-contained measurement notebook

Measures peak VRAM divergence: APIA (fwd-only, `no_grad`) vs LoRA (fwd+bwd+AdamW).
No repo clone — measurement code is embedded here. Enable GPU accelerator first.

Output: `/kaggle/working/vram_results.json` — download and merge into local `spike_results.json`.

In [ ]:
!pip install -q peft accelerate
import torch
print(torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# --- CONFIG ----------------------------------------------------------
MODEL   = 'facebook/opt-1.3b'   # e.g. openai-community/gpt2-large
BATCHES = [1, 4, 8, 16, 32, 64] # sweep upward; OOM is data, not failure
SEQ     = 128
REPS    = 2
TARGETS = ['q_proj', 'v_proj']  # OPT names; GPT-2: ['c_attn','c_proj']
OUT     = '/kaggle/working/vram_results.json'
# ----------------------------------------------------------------------
import time, json
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

dev = 'cuda'
peak = lambda: torch.cuda.max_memory_allocated() / 1e6

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def make_batch(n):
    t = ['the model must adapt to a new domain without forgetting '
         'what it already learned ' * 6] * n
    return tok(t, return_tensors='pt', padding='max_length',
               truncation=True, max_length=SEQ)

def apia_scan():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = AutoModel.from_pretrained(MODEL).to(dev).eval()
    for p in m.parameters(): p.requires_grad_(False)
    w, out = peak(), {}
    for b in BATCHES:
        enc = make_batch(b)
        try:
            t0 = time.perf_counter()
            with torch.no_grad():
                for _ in range(REPS):
                    m(input_ids=enc['input_ids'].to(dev),
                      attention_mask=enc['attention_mask'].to(dev))
            out[b] = dict(w=w, peak=peak(), ms=(time.perf_counter()-t0)/REPS/b*1e3)
            torch.cuda.reset_peak_memory_stats()
        except torch.cuda.OutOfMemoryError:
            out[b] = dict(w=w, peak=-1, ms=-1)
        print(f"apia B={b}: peak={out[b]['peak']:.0f} ms={out[b]['ms']:.2f}", flush=True)
    del m; return out

def lora_scan():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    base.config.pad_token_id = base.config.eos_token_id
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=TARGETS,
                     lora_dropout=0.0, bias='none')
    m = get_peft_model(base, cfg).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    w, out = peak(), {}
    for b in BATCHES:
        enc = make_batch(b)
        lab = torch.zeros(b, dtype=torch.long, device=dev)
        try:
            t0 = time.perf_counter()
            for _ in range(REPS):
                o = m(input_ids=enc['input_ids'].to(dev),
                      attention_mask=enc['attention_mask'].to(dev), labels=lab)
                o.loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
            out[b] = dict(w=w, peak=peak(), ms=(time.perf_counter()-t0)/REPS/b*1e3)
            torch.cuda.reset_peak_memory_stats()
        except torch.cuda.OutOfMemoryError:
            out[b] = dict(w=w, peak=-1, ms=-1)
            torch.cuda.empty_cache()
        print(f"lora B={b}: peak={out[b]['peak']:.0f} ms={out[b]['ms']:.2f}", flush=True)
    del m, opt; return out

In [ ]:
A, L = apia_scan(), lora_scan()
res = {str(b): {'apia': A[b], 'lora': L[b]} for b in BATCHES}
d = {'vram_scaling': {'model': MODEL, 'seq': SEQ,
                    'device': torch.cuda.get_device_name(0),
                    'device_mb': round(torch.cuda.get_device_properties(0).total_memory/1e6),
                    'weights_mb': A[min(BATCHES)]['w'],
                    'by_batch': res}}
json.dump(d, open(OUT, 'w'), indent=2)
print('saved ->', OUT)
for b in BATCHES:
    print(f"B={b:>3}  apia={A[b]['peak']:>7.0f}MB  lora={L[b]['peak']:>7.0f}MB")